In [1]:
import re
from pathlib import Path
import pandas as pd

In [2]:
def parse_whatsapp_chat(file_path: str | Path) -> pd.DataFrame:
    '''
    Lê um arquivo de exportação de chat do WhatsApp (.txt) e retorna
    um DataFrame com as colunas: datetime, sender, message, etc.
    '''
    file_path = Path(file_path)
    
    # Padrão: [M/D/YY, H:M:S AM/PM] Remetente: Mensagem
    pattern = re.compile(
        r'^\[(?P<date>\d{1,2}/\d{1,2}/\d{2,4}),\s*(?P<time>\d{1,2}:\d{2}:\d{2}(?:\s*[APap][Mm])?)\]\s*(?:(?P<sender>[^:]+?):\s+)?(?P<message>.*)$'
    )
    
    records = []
    current_msg = None
    
    with open(file_path, mode='r', encoding='utf-8-sig') as f:
        for line in f:
            line_clean = line.rstrip('\r\n')
            match = pattern.match(line_clean)
            
            if match:
                if current_msg is not None:
                    records.append(current_msg)
                
                data = match.groupdict()
                sender = data['sender']
                msg_text = data['message']
                
                # Mensagens sem remetente separado por dois pontos são avisos do sistema
                if sender is None:
                    sender = 'Sistema'
                
                current_msg = {
                    'date': data['date'],
                    'time': data['time'],
                    'sender': sender.strip(),
                    'message': msg_text,
                }
            else:
                # Continuação da mensagem anterior (caso multilinha)
                if current_msg is not None:
                    current_msg['message'] += '\n' + line_clean
    
    # Adiciona o último registro se existir
    if current_msg is not None:
        records.append(current_msg)
        
    df = pd.DataFrame(records)
    
    if not df.empty:
        # Tratamento de timestamp: substitui espaços múltiplos/especiais (como \u202f) por espaço comum
        datetime_str = (df['date'] + ' ' + df['time']).str.replace(r'\s+', ' ', regex=True)
        
        try:
            df['datetime'] = pd.to_datetime(datetime_str, format='%m/%d/%y %I:%M:%S %p', errors='coerce')
        except Exception:
            df['datetime'] = pd.to_datetime(datetime_str, format='mixed', errors='coerce')
            
        # Ordenação e seleção de colunas
        columns_order = ['datetime', 'sender', 'message', 'date', 'time']
        df = df[[c for c in columns_order if c in df.columns]]
        
    return df

In [3]:
# Localiza o arquivo chat.txt a partir de notebooks/ ou da raiz do projeto
chat_file = Path('../data/raw/chat.txt')

print(f'Carregando arquivo de: {chat_file.resolve()}')
df = parse_whatsapp_chat(chat_file)
print(f'Total de mensagens carregadas: {len(df)}')
df.head(10)

Carregando arquivo de: C:\Users\yanch\OneDrive\Documents\personal-assistant\data\raw\chat.txt
Total de mensagens carregadas: 5213


,datetime,sender,message,date,time
0,2026-08-19 15:36:28,Você,<mensagem de voz omitida>,8/19/26,3:36:28 PM
1,2026-08-19 15:36:35,Você,<mensagem de voz omitida>,8/19/26,3:36:35 PM
2,2026-08-19 15:40:08,Danilo,Comprei as 3 últimas de chocolate,8/19/26,3:40:08 PM
3,2026-08-19 15:40:11,Danilo,É isso mermo,8/19/26,3:40:11 PM
4,2026-08-19 15:41:47,Você,<mensagem de voz omitida>,8/19/26,3:41:47 PM
5,2026-08-19 15:46:05,Você,<mensagem de voz omitida>,8/19/26,3:46:05 PM
6,2026-08-19 15:56:56,Vitu,<imagem ocultada> BORA YAN CHAGAS!!!,8/19/26,3:56:56 PM
7,2026-08-19 15:58:06,Vinio,Tinha que dar metade desse valor pra yan,8/19/26,3:58:06 PM
8,2026-08-19 15:59:25,Vitu,ele daria o valor completo para o JoãoPSX ou u...,8/19/26,3:59:25 PM
9,2026-08-19 15:59:27,Você,.,8/19/26,3:59:27 PM


In [4]:
# Informações gerais do DataFrame
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5213 entries, 0 to 5212
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  5213 non-null   datetime64[us]
 1   sender    5213 non-null   str           
 2   message   5213 non-null   str           
 3   date      5213 non-null   str           
 4   time      5213 non-null   str           
dtypes: datetime64[us](1), str(4)
memory usage: 203.8 KB


In [5]:
# Contagem de mensagens enviadas por cada participante
df['sender'].value_counts()

sender
Você                       2154
Vegeta Petista Feirense     668
Vinio                       646
Danilo                      591
Vitu                        497
Fonsi                       462
Rafael Athaliba              99
Pedrin                       96
Name: count, dtype: int64

In [6]:
# Criação de colunas úteis para análise de texto e mídia e se houve encaminhamento
df['is_media'] = df['message'].str.contains(
    r'<imagem ocultada>|<mensagem de voz omitida>|<figurinha omitida>|<vídeo omitido>|<áudio ocultado>',
    regex=True
)

df['is_foward'] = df['message'].str.contains(
    r'Encaminhada',
    regex=True
)

df.head()

,datetime,sender,message,date,time,is_media,is_foward
0,2026-08-19 15:36:28,Você,<mensagem de voz omitida>,8/19/26,3:36:28 PM,True,False
1,2026-08-19 15:36:35,Você,<mensagem de voz omitida>,8/19/26,3:36:35 PM,True,False
2,2026-08-19 15:40:08,Danilo,Comprei as 3 últimas de chocolate,8/19/26,3:40:08 PM,False,False
3,2026-08-19 15:40:11,Danilo,É isso mermo,8/19/26,3:40:11 PM,False,False
4,2026-08-19 15:41:47,Você,<mensagem de voz omitida>,8/19/26,3:41:47 PM,True,False


In [7]:
# Removendo mensagens de mídia e encaminhadas e colunas desnecessárias
df = df[df['is_media'] == False]
df = df[df['is_foward'] == False]
df = df.drop(columns=['date', 'time', 'is_media', 'is_foward'])

df.head(10)

,datetime,sender,message
2,2026-08-19 15:40:08,Danilo,Comprei as 3 últimas de chocolate
3,2026-08-19 15:40:11,Danilo,É isso mermo
7,2026-08-19 15:58:06,Vinio,Tinha que dar metade desse valor pra yan
8,2026-08-19 15:59:25,Vitu,ele daria o valor completo para o JoãoPSX ou u...
9,2026-08-19 15:59:27,Você,.
10,2026-08-19 15:59:50,Vitu,pense que você está financiando o setup de Vitu
11,2026-08-19 16:00:08,Vitu,dinheiro do MP/ML está todo aplicado no cofrinho
12,2026-08-19 16:00:21,Você,"Oxe, isso não deixa menos pior não"
13,2026-08-19 16:00:45,Vitu,"mas isso deixa\n\nvocê tá dando para um amigo,..."
14,2026-08-19 16:01:13,Você,"Pra mim não muda nada, nos dois casos eu não g..."


In [8]:
# Exportação do DataFrame processado para a pasta data/processed/
output_file = Path('../data/processed/chat_processed.csv')
output_file.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_file, index=False, encoding='utf-8')
print(f'Dados processados exportados com sucesso para: {output_file.resolve()}')
print(f'Total de mensagens salvas: {len(df)}')


Dados processados exportados com sucesso para: C:\Users\yanch\OneDrive\Documents\personal-assistant\data\processed\chat_processed.csv
Total de mensagens salvas: 4483
